# LlamaIndex query engine eval regression testing

Wires LlamaIndex `QueryEngine` evals into pytest with snapshot-based regression detection. Same pattern as the LangChain example, narrower scope.

Uses [evalcheck](https://github.com/Boiga7/evalcheck) for the snapshot diff. MIT, no SaaS account.

In [ ]:
%pip install --quiet llama-index llama-index-llms-openai evalcheck

In [ ]:
from llama_index.core import VectorStoreIndex, Document
from llama_index.llms.openai import OpenAI

documents = [
    Document(text="The Eiffel Tower is in Paris, France."),
    Document(text="The Statue of Liberty is in New York, USA."),
]
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(llm=OpenAI(model="gpt-4o-mini", temperature=0))

## Pytest tests

Each test runs a query, returns the output (and source context for faithfulness), and lets evalcheck score it. Plain `pytest` runs everything.

In [ ]:
%%writefile test_retrieval.py
from evalcheck import EvalOutput, eval, faithfulness, relevance

from __main__ import query_engine

QUESTIONS = {
    "eiffel": "Where is the Eiffel Tower?",
    "liberty": "Where is the Statue of Liberty?",
}

@eval(faithfulness, threshold=0.7)
def test_eiffel_grounded():
    response = query_engine.query(QUESTIONS["eiffel"])
    context = "\n".join(n.node.text for n in response.source_nodes)
    return EvalOutput(output=str(response), context=context)

@eval(relevance, threshold=0.7)
def test_liberty_relevant():
    response = query_engine.query(QUESTIONS["liberty"])
    return EvalOutput(output=str(response), input=QUESTIONS["liberty"])

In [ ]:
!pytest test_retrieval.py -v
!evalcheck snapshot --update     # bless the first run as baseline
# git add .evalcheck && git commit -m 'chore: bless eval baseline'
# Future runs fail if relevance/faithfulness drops > 0.05 below baseline.

## Combining with LlamaIndex's own evaluators

If you'd rather use LlamaIndex's `FaithfulnessEvaluator`, wrap it as a custom evalcheck metric:

```python
from llama_index.core.evaluation import FaithfulnessEvaluator
from evalcheck import eval, EvalOutput

li_eval = FaithfulnessEvaluator(llm=OpenAI(model='gpt-4o-mini'))

def li_faithfulness(output: str, context: str) -> float:
    result = li_eval.evaluate(response=output, contexts=[context])
    return float(result.passing)

@eval(li_faithfulness, threshold=0.7)
def test_with_llamaindex_evaluator():
    response = query_engine.query(...)
    return EvalOutput(output=str(response), context=...)
```

## More

- [evalcheck repo](https://github.com/Boiga7/evalcheck)
- [Comparisons vs deepeval and others](https://github.com/Boiga7/evalcheck/tree/main/docs/comparisons)